# Worked Example: ACC–Frontal Connectivity

## Goal
Spectral connectivity on real feedback epochs with surrogate null. See 11_advanced_utility_interoperability.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from mne_connectivity import spectral_connectivity_epochs
from LFPAnalysis import load_lfp, oscillation_utils
from LFPAnalysis.config import LoadConfig

beh = pd.read_csv(Path('../../data/sample_beh.csv'))
epochs = load_lfp(LoadConfig(path=Path('../../data/sample_feedback_start-epo.fif'), file_format='mne'))
epochs.metadata = beh[['reward', 'rpe']]
epochs_sub = epochs.copy().pick(['racas1-racas2', 'rmolf5-rmolf6'])
con = spectral_connectivity_epochs(
    epochs_sub, method='coh', mode='multitaper', fmin=13, fmax=30, faverage=True, verbose=False
)
coh_value = float(con.get_data()[0, 0])
print(f'Beta coherence ACC–frontal: {coh_value:.3f}')

In [ ]:
seed_data = epochs_sub.get_data()[:, 0, :]
surr = oscillation_utils.make_surrogate_arrays(
    seed_data, method='swap_epochs', n_shuffles=50, rng_seed=42, return_generator=False
)
surr_coh = [float(np.corrcoef(surr[i], epochs_sub.get_data()[:, 1, :].mean(axis=0))[0, 1]) for i in range(min(10, len(surr)))]
fig, ax = plt.subplots(figsize=(5, 3))
ax.hist(surr_coh, bins=15, color='0.7', label='surrogate (approx)')
ax.axvline(coh_value, color='r', lw=2, label='observed')
ax.set(xlabel='Coupling proxy', title='Surrogate null (illustrative)')
ax.legend()
fig.tight_layout()
plt.show()

## Next step

11_advanced_utility_interoperability for full connectivity API. Chapter 10b for statistics.